# Lab 12 · LlamaIndex ingestion into pgvector
## Goal
Build and change an ingestion pipeline, generate real embeddings, and persist versioned documents. Prerequisites: Python functions/dictionaries, introductory SQL, Lab 6. Plan for 120 minutes after environment setup: 20 exploration, 45 implementation, 30 experiments, 25 tests/report. Timing is a teaching estimate.

Deliver an ingestion manifest and evidence that re-ingestion, updates, and deletion behave correctly. You will use LlamaIndex `Document`, `SentenceSplitter`, and `IngestionPipeline`; PostgreSQL transactions own persistence in this implementation.

## Setup
Complete the track README preflight first: install this track's pinned environment and editable package; start PostgreSQL/pgvector with Docker Compose; download the two pinned model snapshots; seed the invented course corpus. Select that Python environment as the kernel. These notebooks need the **whole `labs/rag` folder**, not a standalone upload. They use a real database and local neural models. Missing services or models are errors, not silently replaced by mocks.

Run worked cells first, then implement TODOs, set `RUN_EXERCISES=True`, restart, and run all. Instructor copies enable those checks. `RUN_GENERATION` stays false unless you deliberately enable the separately documented Gemini request. Offline here means no inference API calls after package/model downloads; PostgreSQL is still a real local service. All data are invented and English-language. Never use this teaching database or its public example keys for real student records.

In [1]:
import json
from pathlib import Path
from importlib.metadata import version
import numpy as np
from raglab.config import ROOT, DIMENSION
from raglab import db, models
from raglab.ingestion import corpus, chunk_documents, make_pipeline
RUN_EXERCISES = True
RUN_GENERATION = False
print({p:version(p) for p in ("llama-index-core","pgvector","psycopg","fastapi","sentence-transformers")})
with db.connect() as conn:
    print(conn.execute("SELECT extversion FROM pg_extension WHERE extname='vector'").fetchone())

{'llama-index-core': '0.14.25', 'pgvector': '0.5.0', 'psycopg': '3.3.6', 'fastapi': '0.142.4', 'sentence-transformers': '6.1.0'}
{'extversion': '0.8.1'}


## Steps
### 1. Inspect actual chunks and embeddings
The fixture corpus contains two tenants. Chunk IDs incorporate tenant, document, version, ordinal, and content. Metadata stays attached to chunks but is excluded from the embedding text in this pipeline. Inspect the consequence of excluding the title: retrieval may lose a useful signal. The embedding snapshot is a compact teaching baseline, not a claim of current model superiority.

In [2]:
records=corpus()
alpha=[r for r in records if r["tenant"]=="alpha"]
chunks=chunk_documents(alpha[:2])
vectors=models.embed([c["text"] for c in chunks])
assert vectors.shape==(len(chunks),384)
assert np.allclose(np.linalg.norm(vectors,axis=1),1,atol=1e-5)
assert len({c["chunk_id"] for c in chunks})==len(chunks)
print({"documents":len(records),"chunks_in_example":len(chunks),"vector_shape":vectors.shape})
print({k:chunks[0][k] for k in ("doc_id","version","source","text")})

/private/tmp/course-rag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13750.07it/s]

{'documents': 17, 'chunks_in_example': 2, 'vector_shape': (2, 384)}
{'doc_id': 'camera-loans', 'version': 1, 'source': 'course://alpha/camera-loans', 'text': 'Cameras can be borrowed for three days. Bring a valid student card to collect the camera. Overnight loans count toward the three-day period. Return the camera to the media desk before closing.'}


### 2. Inspect chunk boundaries
Use actual text, not just a chunk count, to decide whether an exception remains with its rule. Smaller chunks may lose context; overlapping chunks may repeat evidence. Do not tune chunking using the public assessment split. A chunk-size parameter is measured in the splitter's tokenizer units, not characters.

In [3]:
for size,overlap in [(32,0),(64,8),(96,12)]:
    sample=chunk_documents([alpha[1]],make_pipeline(size,overlap))
    print({"chunk_size":size,"overlap":overlap,"chunks":len(sample)})
    print([c["text"] for c in sample])

Metadata length (0) is close to chunk size (32). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.


{'chunk_size': 32, 'overlap': 0, 'chunks': 2}
['Camera loans can be extended by two days only if another student has not reserved the equipment. Request an extension before the original due date.', 'The media desk must approve an extension; submitting a request does not automatically extend a loan.']
{'chunk_size': 64, 'overlap': 8, 'chunks': 1}
['Camera loans can be extended by two days only if another student has not reserved the equipment. Request an extension before the original due date. The media desk must approve an extension; submitting a request does not automatically extend a loan.']
{'chunk_size': 96, 'overlap': 12, 'chunks': 1}
['Camera loans can be extended by two days only if another student has not reserved the equipment. Request an extension before the original due date. The media desk must approve an extension; submitting a request does not automatically extend a loan.']


### 3. TODO: configure and inspect a LlamaIndex pipeline (30 points)
Implement `student_chunks(records, size, overlap)` by constructing `IngestionPipeline` with `SentenceSplitter`, then passing it to `chunk_documents`. Enforce the same size/overlap contract as `make_pipeline` (size 32–256, overlap 0..size-1, integers excluding booleans). Do not call `make_pipeline` in your implementation. Preserve metadata and return stable chunk IDs.

In [4]:
def student_chunks(records,size,overlap):
    from llama_index.core.ingestion import IngestionPipeline
    from llama_index.core.node_parser import SentenceSplitter
    if type(size) is not int or type(overlap) is not int or not 32<=size<=256 or not 0<=overlap<size:
        raise ValueError("invalid chunk parameters")
    pipeline=IngestionPipeline(transformations=[SentenceSplitter(chunk_size=size,chunk_overlap=overlap)])
    return chunk_documents(records,pipeline)

In [5]:
if RUN_EXERCISES:
    assert student_chunks([],64,8)==[]
    a=student_chunks(alpha[:2],64,8)
    b=student_chunks(alpha[:2],64,8)
    assert [c["chunk_id"] for c in a]==[c["chunk_id"] for c in b]
    assert all(c["tenant"]=="alpha" and c["source"].startswith("course://") for c in a)
    for args in [(True,0),(31,0),(64,64),(64,-1)]:
        try: student_chunks(alpha[:1],*args)
        except ValueError: pass
        else: raise AssertionError("invalid chunk configuration accepted")

### 4. TODO: persist, re-ingest, and update (40 points)
Implement `ingest_one(record)` using your splitter (64,8), `models.embed`, and `db.replace_document`. Return the number of chunks. Read `raglab/db.py`: identify the row lock, transaction boundary, and cascading foreign key. The helper replaces only the specified tenant/document; it never truncates the corpus. Tests use an isolated teaching tenant and remove only their own document.

In [6]:
def ingest_one(record):
    parts=student_chunks([record],64,8)
    vectors=models.embed([p["text"] for p in parts])
    db.replace_document(record,parts,vectors)
    return len(parts)

In [7]:
if RUN_EXERCISES:
    record=dict(alpha[0],tenant="exercise12",doc_id="temporary",source="course://exercise12/temporary")
    try:
        count=ingest_one(record)
        assert ingest_one(record)==count
        with db.connect() as conn:
            actual=conn.execute("SELECT count(*) AS n FROM rag_course.chunks WHERE tenant=%s AND doc_id=%s",("exercise12","temporary")).fetchone()["n"]
        assert actual==count
        updated=dict(record,version=2,body="Camera loans last four days in this isolated exercise.")
        ingest_one(updated)
        try: ingest_one(record)
        except ValueError: pass
        else: raise AssertionError("stale update accepted")
    finally:
        db.delete_document("exercise12","temporary")
    print("Ingestion, idempotent replay, version update, and cleanup checked")

Ingestion, idempotent replay, version update, and cleanup checked


## Checks and submission
Save a small manifest: document count, chunk count, splitter settings, embedding snapshot revision, vector dimension, and source hashes. Add tests for duplicate document IDs, a blank body, and a database insert failure that must roll back the replacement. Compare two chunking configurations on development questions. Record a failure example even if aggregate scores are equal.

Rubric: pipeline 30, persistence 40, added tests 15, reproducible manifest and explanation 15. LlamaIndex pipeline caching is not the same as PostgreSQL upsert/version policy; identify which layer owns each operation.

## Next Steps
Lab 13 examines how pgvector searches the stored vectors. The student should be able to locate a returned chunk in the original document before treating it as evidence.

References checked 2026-10-07: [LlamaIndex ingestion](https://developers.llamaindex.ai/python/framework/module_guides/loading/ingestion_pipeline/), [pgvector storage and indexes](https://github.com/pgvector/pgvector), [embedding model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).